# Démo Complète: Pipeline Multi-Batch avec Nessie & Iceberg

**Scénario de démonstration d'entretien**

Cette démo simule un pipeline de données en production qui reçoit des fichiers tout au long de la journée.

## Plan de la démo

| Étape | Action | Ce qu'on montre |
|-------|--------|-----------------|
| 1 | **Batch 1** (matin) | Ingestion initiale, création des tables |
| 2 | **Batch 2** (midi) | Append, mise à jour incrémentale |
| 3 | **Batch 3** (après-midi) | Append, évolution des données |
| 4 | **Time Travel** | Requêter les données à différents moments |
| 5 | **Rollback** | Revenir à un état précédent |
| 6 | **Branching** | Développement isolé sur branche dev |

**Prérequis**:
- Nessie démarré: `docker run -p 19120:19120 ghcr.io/projectnessie/nessie:latest`
- Données uploadées sur S3: `python scripts/upload_to_s3.py`

---
## INITIALISATION

In [1]:
import sys
import os

project_root = os.path.abspath("..")
src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

from lakehouse.spark_session import get_spark
from lakehouse.settings import AWS_BUCKET
from lakehouse.settings import NESSIE_URI

spark = get_spark("demo-complete")

print("""
╔══════════════════════════════════════════════════════════════╗
║              DÉMO LAKEHOUSE - INITIALISATION                 ║
╚══════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════╗
║              DÉMO LAKEHOUSE - INITIALISATION                 ║
╚══════════════════════════════════════════════════════════════╝



In [2]:
# S'assurer qu'on est sur la branche main
spark.conf.set("spark.sql.catalog.nessie.ref", "main")

# Créer les namespaces
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.bronze")
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.silver")
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.gold")

print("✅ Sur la branche 'main'")
print("✅ Namespaces Nessie prêts")
print(f"Bucket S3: {AWS_BUCKET}")

✅ Sur la branche 'main'
✅ Namespaces Nessie prêts
Bucket S3: iceberg-nessie-lakehouse-demo


---
## BATCH 1 - PREMIÈRE INGESTION (MATIN)

**Scénario**: C'est le matin, le premier fichier de ventes arrive.

Fichier: `sales_batch_1.csv` (3,364 enregistrements)

In [3]:
from pyspark.sql.functions import current_timestamp, current_date, lit

# Supprimer les tables si elles existent (pour une démo propre)
spark.sql("DROP TABLE IF EXISTS nessie.bronze.sales")
spark.sql("DROP TABLE IF EXISTS nessie.silver.sales")
spark.sql("DROP TABLE IF EXISTS nessie.gold.sales_by_category_region")

print("🗑️ Tables existantes supprimées !")

🗑️ Tables existantes supprimées !


In [4]:
# Lire le premier batch depuis S3
batch1_path = f"s3a://{AWS_BUCKET}/raw/batches/sales_batch_001.csv"

batch1_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(batch1_path)
)

batch1_count = batch1_raw.count()
print(f"📥 BATCH 1 lu: {batch1_count:,} enregistrements")

📥 BATCH 1 lu: 3,364 enregistrements


In [5]:
# Enrichir et créer la table Bronze
from pyspark.sql.functions import col

batch1_bronze = (
    batch1_raw
    .withColumn("ingestion_date", current_date())
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", lit("sales_batch_1.csv"))
    .withColumn("source_system", lit("sales_system"))
    .withColumn("batch_id", lit("batch_1"))
)

batch1_bronze.writeTo("nessie.bronze.sales").using("iceberg").partitionedBy(col("ingestion_date")).create()

print("✅ BATCH 1 → Table Bronze créée")
print(f"Enregistrements: {batch1_count:,}")

✅ BATCH 1 → Table Bronze créée
Enregistrements: 3,364


In [6]:
# Transformer vers Silver
batch1_silver = (
    batch1_bronze
    .select(
        col("Order ID").alias("order_id"),
        col("Order Date").alias("order_date"),
        col("Ship Date").alias("ship_date"),
        col("Ship Mode").alias("ship_mode"),
        col("Customer ID").alias("customer_id"),
        col("Customer Name").alias("customer_name"),
        col("Segment").alias("segment"),
        col("Country").alias("country"),
        col("City").alias("city"),
        col("State").alias("state"),
        col("Postal Code").alias("postal_code"),
        col("Region").alias("region"),
        col("Product ID").alias("product_id"),
        col("Category").alias("category"),
        col("Sub-Category").alias("sub_category"),
        col("Product Name").alias("product_name"),
        col("Sales").cast("double").alias("sales"),
        col("Quantity").cast("int").alias("quantity"),
        col("Discount").cast("double").alias("discount"),
        col("Profit").cast("double").alias("profit"),
        col("ingestion_date"),
        col("ingestion_ts"),
        col("batch_id")
    )
    .filter(col("order_id").isNotNull())
    .filter(col("product_id").isNotNull())
    .filter(col("sales").isNotNull())
    .dropDuplicates(["order_id", "product_id"])
)

batch1_silver.writeTo("nessie.silver.sales").using("iceberg").create()

silver_count = batch1_silver.count()
print("✅ BATCH 1 → Table Silver créée")
print(f"Enregistrements: {silver_count:,}")

✅ BATCH 1 → Table Silver créée
Enregistrements: 3,260


In [7]:
# Créer l'agrégat Gold
from pyspark.sql.functions import sum as spark_sum, round as spark_round, count

batch1_gold = (
    batch1_silver
    .groupBy("category", "region")
    .agg(
        spark_round(spark_sum("sales"), 2).alias("total_sales"),
        spark_round(spark_sum("profit"), 2).alias("total_profit"),
        spark_sum("quantity").alias("total_quantity"),
        count("order_id").alias("order_count")
    )
)

batch1_gold.writeTo("nessie.gold.sales_by_category_region").using("iceberg").create()

gold_count = batch1_gold.count()
print("✅ BATCH 1 → Table Gold créée")
print(f"Enregistrements: {gold_count:,}")

✅ BATCH 1 → Table Gold créée
Enregistrements: 12


In [8]:
# Afficher les résultats du Batch 1
print("\n" + "="*60)
print("RÉSULTATS APRÈS BATCH 1")
print("="*60)
print(f"Bronze: {spark.sql('SELECT COUNT(*) FROM nessie.bronze.sales').first()[0]:,} enregistrements")
print(f"Silver: {spark.sql('SELECT COUNT(*) FROM nessie.silver.sales').first()[0]:,} enregistrements")
print(f"Gold: {spark.sql('SELECT COUNT(*) FROM nessie.gold.sales_by_category_region').first()[0]:,} agrégats")

print("\nTop 5 ventes par catégorie/région:")
spark.sql("""
    SELECT category, region, total_sales, order_count
    FROM nessie.gold.sales_by_category_region
    ORDER BY total_sales DESC
    LIMIT 5
""").show(truncate=False)


RÉSULTATS APRÈS BATCH 1
Bronze: 3,364 enregistrements
Silver: 3,260 enregistrements
Gold: 12 agrégats

Top 5 ventes par catégorie/région:
+---------------+------+-----------+-----------+
|category       |region|total_sales|order_count|
+---------------+------+-----------+-----------+
|Furniture      |West  |84314.97   |234        |
|Furniture      |East  |76048.63   |192        |
|Office Supplies|West  |75146.26   |598        |
|Technology     |West  |71744.79   |191        |
|Technology     |East  |70360.68   |167        |
+---------------+------+-----------+-----------+



In [9]:
# Sauvegarder le snapshot ID du Batch 1 pour le time travel
batch1_snapshot_bronze = spark.sql("""
    SELECT snapshot_id 
    FROM nessie.bronze.sales.snapshots 
    ORDER BY committed_at DESC 
    LIMIT 1
""").first()[0]

batch1_snapshot_silver = spark.sql("""
    SELECT snapshot_id 
    FROM nessie.silver.sales.snapshots 
    ORDER BY committed_at DESC 
    LIMIT 1
""").first()[0]

batch1_snapshot_gold = spark.sql("""
    SELECT snapshot_id 
    FROM nessie.gold.sales_by_category_region.snapshots 
    ORDER BY committed_at DESC 
    LIMIT 1
""").first()[0]

print(f"📌 Snapshot ID Bronze (Batch 1): {batch1_snapshot_bronze}")
print(f"📌 Snapshot ID Silver (Batch 1): {batch1_snapshot_silver}")
print(f"📌 Snapshot ID Gold (Batch 1):   {batch1_snapshot_gold}")
print("(Sauvegardés pour le time travel)")

📌 Snapshot ID Bronze (Batch 1): 2596944187902129844
📌 Snapshot ID Silver (Batch 1): 2194611863969289196
📌 Snapshot ID Gold (Batch 1):   8572448889125248201
(Sauvegardés pour le time travel)


---
## BATCH 2 - DEUXIÈME INGESTION (MIDI)

**Scénario**: C'est le midi, un nouveau fichier de ventes arrive.

Fichier: `sales_batch_2.csv` (3,501 enregistrements)

**Point clé à montrer**: Les nouvelles données sont **ajoutées** aux existantes (append mode).

In [10]:
# Lire le deuxième batch depuis S3
batch2_path = f"s3a://{AWS_BUCKET}/raw/batches/sales_batch_002.csv"

batch2_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(batch2_path)
)

batch2_count = batch2_raw.count()
print(f"📥 BATCH 2 lu: {batch2_count:,} enregistrements")

📥 BATCH 2 lu: 3,501 enregistrements


In [11]:
# Enrichir et AJOUTER à la table Bronze (APPEND)
batch2_bronze = (
    batch2_raw
    .withColumn("ingestion_date", current_date())
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", lit("sales_batch_2.csv"))
    .withColumn("source_system", lit("sales_system"))
    .withColumn("batch_id", lit("batch_2"))
)

# Mode APPEND - les données sont ajoutées aux existantes!
batch2_bronze.writeTo("nessie.bronze.sales").append()

new_bronze_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print("✅ BATCH 2 → Ajouté à la table Bronze (APPEND)")
print(f"Nouveau total Bronze: {new_bronze_count:,} (+{new_bronze_count - batch1_count})")

✅ BATCH 2 → Ajouté à la table Bronze (APPEND)
Nouveau total Bronze: 6,865 (+3501)


In [12]:
# Transformer et AJOUTER à Silver
batch2_silver = (
    batch2_bronze
    .select(
        col("Order ID").alias("order_id"),
        col("Order Date").alias("order_date"),
        col("Ship Date").alias("ship_date"),
        col("Ship Mode").alias("ship_mode"),
        col("Customer ID").alias("customer_id"),
        col("Customer Name").alias("customer_name"),
        col("Segment").alias("segment"),
        col("Country").alias("country"),
        col("City").alias("city"),
        col("State").alias("state"),
        col("Postal Code").alias("postal_code"),
        col("Region").alias("region"),
        col("Product ID").alias("product_id"),
        col("Category").alias("category"),
        col("Sub-Category").alias("sub_category"),
        col("Product Name").alias("product_name"),
        col("Sales").cast("double").alias("sales"),
        col("Quantity").cast("int").alias("quantity"),
        col("Discount").cast("double").alias("discount"),
        col("Profit").cast("double").alias("profit"),
        col("ingestion_date"),
        col("ingestion_ts"),
        col("batch_id")
    )
    .filter(col("order_id").isNotNull())
    .filter(col("product_id").isNotNull())
    .filter(col("sales").isNotNull())
    .dropDuplicates(["order_id", "product_id"])
)

batch2_silver.writeTo("nessie.silver.sales").append()

new_silver_count = spark.sql("SELECT COUNT(*) FROM nessie.silver.sales").first()[0]
print("✅ BATCH 2 → Ajouté à la table Silver (APPEND)")
print(f"Nouveau total Silver: {new_silver_count:,} (+{new_silver_count - silver_count})")

✅ BATCH 2 → Ajouté à la table Silver (APPEND)
Nouveau total Silver: 6,587 (+3327)


In [13]:
# Recalculer et remplacer Gold (table de remplacement)
all_silver = spark.table("nessie.silver.sales")

new_gold = (
    all_silver
    .groupBy("category", "region")
    .agg(
        spark_round(spark_sum("sales"), 2).alias("total_sales"),
        spark_round(spark_sum("profit"), 2).alias("total_profit"),
        spark_sum("quantity").alias("total_quantity"),
        count("order_id").alias("order_count")
    )
)

new_gold.writeTo("nessie.gold.sales_by_category_region").using("iceberg").createOrReplace()

print("✅ Gold recalculé avec toutes les données de Silver")
print(f"Agrégats: {new_gold.count():,}")

✅ Gold recalculé avec toutes les données de Silver
Agrégats: 12


In [14]:
# Afficher les résultats après Batch 2
print("\n" + "="*60)
print("RÉSULTATS APRÈS BATCH 2")
print("="*60)
print(f"Bronze: {spark.sql('SELECT COUNT(*) FROM nessie.bronze.sales').first()[0]:,} enregistrements")
print(f"Silver: {spark.sql('SELECT COUNT(*) FROM nessie.silver.sales').first()[0]:,} enregistrements")
print(f"Gold: {spark.sql('SELECT COUNT(*) FROM nessie.gold.sales_by_category_region').first()[0]:,} agrégats")

print("\nTop 5 ventes par catégorie/région:")
spark.sql("""
    SELECT category, region, total_sales, order_count
    FROM nessie.gold.sales_by_category_region
    ORDER BY total_sales DESC
    LIMIT 5
""").show(truncate=False)


RÉSULTATS APRÈS BATCH 2
Bronze: 6,865 enregistrements
Silver: 6,587 enregistrements
Gold: 12 agrégats

Top 5 ventes par catégorie/région:
+---------------+------+-----------+-----------+
|category       |region|total_sales|order_count|
+---------------+------+-----------+-----------+
|Furniture      |West  |180579.27  |475        |
|Technology     |East  |173185.88  |342        |
|Technology     |West  |154526.5   |382        |
|Furniture      |East  |145835.02  |397        |
|Office Supplies|East  |139495.68  |1157       |
+---------------+------+-----------+-----------+



In [15]:
# Sauvegarder le snapshot ID du Batch 2
batch2_snapshot_bronze = spark.sql("""
    SELECT snapshot_id 
    FROM nessie.bronze.sales.snapshots 
    ORDER BY committed_at DESC 
    LIMIT 1
""").first()[0]

batch2_snapshot_silver = spark.sql("""
    SELECT snapshot_id 
    FROM nessie.silver.sales.snapshots 
    ORDER BY committed_at DESC 
    LIMIT 1
""").first()[0]

batch2_snapshot_gold = spark.sql("""
    SELECT snapshot_id 
    FROM nessie.gold.sales_by_category_region.snapshots 
    ORDER BY committed_at DESC 
    LIMIT 1
""").first()[0]

print(f"📌 Snapshot ID Bronze (Batch 2): {batch2_snapshot_bronze}")
print(f"📌 Snapshot ID Silver (Batch 2): {batch2_snapshot_silver}")
print(f"📌 Snapshot ID Gold (Batch 2):   {batch2_snapshot_gold}")

📌 Snapshot ID Bronze (Batch 2): 2127437416950433543
📌 Snapshot ID Silver (Batch 2): 8072671431636041107
📌 Snapshot ID Gold (Batch 2):   8406319559651539367


---
## BATCH 3 - TROISIÈME INGESTION (APRÈS-MIDI)

**Scénario**: C'est l'après-midi, un dernier fichier de ventes arrive.

Fichier: `sales_batch_3.csv` (3,500 enregistrements)

In [16]:
# Lire le troisième batch depuis S3
batch3_path = f"s3a://{AWS_BUCKET}/raw/batches/sales_batch_003.csv"

batch3_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(batch3_path)
)

batch3_count = batch3_raw.count()
print(f"📥 BATCH 3 lu: {batch3_count:,} enregistrements")

📥 BATCH 3 lu: 3,500 enregistrements


In [17]:
# Enrichir et AJOUTER à Bronze
batch3_bronze = (
    batch3_raw
    .withColumn("ingestion_date", current_date())
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", lit("sales_batch_3.csv"))
    .withColumn("source_system", lit("sales_system"))
    .withColumn("batch_id", lit("batch_3"))
)

batch3_bronze.writeTo("nessie.bronze.sales").append()

final_bronze_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print("✅ BATCH 3 → Ajouté à Bronze")
print(f"Total Bronze: {final_bronze_count:,}")

✅ BATCH 3 → Ajouté à Bronze
Total Bronze: 10,365


In [18]:
# Transformer et AJOUTER à Silver
batch3_silver = (
    batch3_bronze
    .select(
        col("Order ID").alias("order_id"),
        col("Order Date").alias("order_date"),
        col("Ship Date").alias("ship_date"),
        col("Ship Mode").alias("ship_mode"),
        col("Customer ID").alias("customer_id"),
        col("Customer Name").alias("customer_name"),
        col("Segment").alias("segment"),
        col("Country").alias("country"),
        col("City").alias("city"),
        col("State").alias("state"),
        col("Postal Code").alias("postal_code"),
        col("Region").alias("region"),
        col("Product ID").alias("product_id"),
        col("Category").alias("category"),
        col("Sub-Category").alias("sub_category"),
        col("Product Name").alias("product_name"),
        col("Sales").cast("double").alias("sales"),
        col("Quantity").cast("int").alias("quantity"),
        col("Discount").cast("double").alias("discount"),
        col("Profit").cast("double").alias("profit"),
        col("ingestion_date"),
        col("ingestion_ts"),
        col("batch_id")
    )
    .filter(col("order_id").isNotNull())
    .filter(col("product_id").isNotNull())
    .filter(col("sales").isNotNull())
    .dropDuplicates(["order_id", "product_id"])
)

batch3_silver.writeTo("nessie.silver.sales").append()

final_silver_count = spark.sql("SELECT COUNT(*) FROM nessie.silver.sales").first()[0]
print("✅ BATCH 3 → Ajouté à Silver")
print(f"Total Silver: {final_silver_count:,}")

✅ BATCH 3 → Ajouté à Silver
Total Silver: 9,917


In [19]:
# Recalculer Gold
all_silver_final = spark.table("nessie.silver.sales")

final_gold = (
    all_silver_final
    .groupBy("category", "region")
    .agg(
        spark_round(spark_sum("sales"), 2).alias("total_sales"),
        spark_round(spark_sum("profit"), 2).alias("total_profit"),
        spark_sum("quantity").alias("total_quantity"),
        count("order_id").alias("order_count")
    )
)

final_gold.writeTo("nessie.gold.sales_by_category_region").using("iceberg").createOrReplace()

print("✅ Gold recalculé final")
print(f"Agrégats: {final_gold.count():,}")

✅ Gold recalculé final
Agrégats: 12


In [20]:
# Afficher les résultats finaux
print("\n" + "="*60)
print("RÉSULTATS APRÈS BATCH 3 (FINAL)")
print("="*60)
print(f"Bronze: {final_bronze_count:,} enregistrements")
print(f"Silver: {final_silver_count:,} enregistrements")
print(f"Gold: {final_gold.count():,} agrégats")

print("\nTop 5 ventes par catégorie/région:")
spark.sql("""
    SELECT category, region, total_sales, order_count
    FROM nessie.gold.sales_by_category_region
    ORDER BY total_sales DESC
    LIMIT 5
""").show(truncate=False)


RÉSULTATS APRÈS BATCH 3 (FINAL)
Bronze: 10,365 enregistrements
Silver: 9,917 enregistrements
Gold: 12 agrégats

Top 5 ventes par catégorie/région:
+---------------+------+-----------+-----------+
|category       |region|total_sales|order_count|
+---------------+------+-----------+-----------+
|Technology     |East  |260324.31  |528        |
|Furniture      |West  |252284.98  |706        |
|Technology     |West  |250685.68  |595        |
|Office Supplies|West  |220154.6   |1883       |
|Furniture      |East  |206859.67  |595        |
+---------------+------+-----------+-----------+



---
## TIME TRAVEL - VOYAGE DANS LE TEMPS

**Point clé**: Avec Iceberg, on peut requêter les données telles qu'elles étaient à n'importe quel moment passé !

In [21]:
# Afficher l'historique complet des snapshots
print("=== Historique des Snapshots Bronze ===")
spark.sql("""
    SELECT 
        made_current_at,
        snapshot_id,
        parent_id
    FROM nessie.bronze.sales.history
    ORDER BY made_current_at
""").show(truncate=False)

=== Historique des Snapshots Bronze ===
+-----------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |
+-----------------------+-------------------+-------------------+
|2026-03-15 16:16:15.59 |2596944187902129844|NULL               |
|2026-03-15 16:16:22.21 |2127437416950433543|2596944187902129844|
|2026-03-15 16:16:26.955|490886184201222251 |2127437416950433543|
+-----------------------+-------------------+-------------------+



In [22]:
# Time Travel 1: Requêter les données du Batch 1 uniquement
print("=== Time Travel: Données après Batch 1 ===")
count_after_batch1 = spark.sql(f"""
    SELECT COUNT(*) as cnt 
    FROM nessie.bronze.sales VERSION AS OF '{batch1_snapshot_bronze}'
""").first()[0]

print(f"Nombre d'enregistrements après Batch 1: {count_after_batch1:,}")

spark.sql(f"""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.bronze.sales VERSION AS OF '{batch1_snapshot_bronze}'
    GROUP BY batch_id
    ORDER BY batch_id
""").show()

=== Time Travel: Données après Batch 1 ===
Nombre d'enregistrements après Batch 1: 3,364
+--------+-----+
|batch_id|count|
+--------+-----+
| batch_1| 3364|
+--------+-----+



In [23]:
# Time Travel 2: Requêter les données du Batch 2
print("=== Time Travel: Données après Batch 2 ===")
count_after_batch2 = spark.sql(f"""
    SELECT COUNT(*) as cnt 
    FROM nessie.bronze.sales VERSION AS OF '{batch2_snapshot_bronze}'
""").first()[0]

print(f"Nombre d'enregistrements après Batch 2: {count_after_batch2:,}")

spark.sql(f"""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.bronze.sales VERSION AS OF '{batch2_snapshot_bronze}'
    GROUP BY batch_id
    ORDER BY batch_id
""").show()

=== Time Travel: Données après Batch 2 ===
Nombre d'enregistrements après Batch 2: 6,865
+--------+-----+
|batch_id|count|
+--------+-----+
| batch_1| 3364|
| batch_2| 3501|
+--------+-----+



In [24]:
# Time Travel 3: Données actuelles (après Batch 3)
print("=== Données actuelles (après Batch 3) ===")
spark.sql("""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.bronze.sales
    GROUP BY batch_id
    ORDER BY batch_id
""").show()

print("🔍 Observation: Chaque batch crée un nouveau snapshot!")
print("On peut revenir à n'importe quel moment dans le temps.")

=== Données actuelles (après Batch 3) ===
+--------+-----+
|batch_id|count|
+--------+-----+
| batch_1| 3364|
| batch_2| 3501|
| batch_3| 3500|
+--------+-----+

🔍 Observation: Chaque batch crée un nouveau snapshot!
On peut revenir à n'importe quel moment dans le temps.


In [25]:
# Comparaison des agrégats Gold dans le temps
print("=== Évolution des ventes Technology/East ===")

# Après Batch 1 (utiliser le snapshot GOLD)
sales_batch1 = spark.sql(f"""
    SELECT total_sales 
    FROM nessie.gold.sales_by_category_region VERSION AS OF '{batch1_snapshot_gold}'
    WHERE category = 'Technology' AND region = 'East'
""").first()[0]

# Actuel
sales_current = spark.sql("""
    SELECT total_sales 
    FROM nessie.gold.sales_by_category_region
    WHERE category = 'Technology' AND region = 'East'
""").first()[0]

print(f"Ventes Technology/East après Batch 1: {sales_batch1:,.2f}")
print(f"Ventes Technology/East actuelles: {sales_current:,.2f}")
print(f"Augmentation: {sales_current - sales_batch1:,.2f}")

=== Évolution des ventes Technology/East ===
Ventes Technology/East après Batch 1: 70,360.68
Ventes Technology/East actuelles: 260,324.31
Augmentation: 189,963.63


In [26]:
# État actuel avant rollback
print("=== ÉTAT ACTUEL (après Batch 3) ===")

bronze_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
silver_count = spark.sql("SELECT COUNT(*) FROM nessie.silver.sales").first()[0]
gold_count = spark.sql("SELECT COUNT(*) FROM nessie.gold.sales_by_category_region").first()[0]

print(f"Bronze: {bronze_count:,} enregistrements")
print(f"Silver: {silver_count:,} enregistrements")
print(f"Gold: {gold_count:,} agrégats")

# Snapshot actuel
current_snapshot = spark.sql("""
    SELECT snapshot_id 
    FROM nessie.bronze.sales.snapshots 
    ORDER BY committed_at DESC 
    LIMIT 1
""").first()[0]

print(f"\nSnapshot actuel: {current_snapshot}")

=== ÉTAT ACTUEL (après Batch 3) ===
Bronze: 10,365 enregistrements
Silver: 9,917 enregistrements
Gold: 12 agrégats

Snapshot actuel: 490886184201222251


In [27]:
# ROLLBACK: Revenir à l'état après Batch 2
print("=== ROLLBACK vers l'état après Batch 2 ===")
print(f"Snapshot cible: {batch2_snapshot_bronze}")
print("\n🔙 Rollback de la table Bronze vers le snapshot après Batch 2...")

# Méthode de rollback avec Iceberg
bronze_batch2 = spark.sql(f"""
    SELECT * FROM nessie.bronze.sales VERSION AS OF '{batch2_snapshot_bronze}'
""")

# Compter
count_batch2 = bronze_batch2.count()
print(f"Données à restaurer: {count_batch2:,} enregistrements")

# Rollback: remplacer le contenu de la table
bronze_batch2.writeTo("nessie.bronze.sales").using("iceberg").createOrReplace()

print("\n✅ Rollback Bronze terminé!")

=== ROLLBACK vers l'état après Batch 2 ===
Snapshot cible: 2127437416950433543

🔙 Rollback de la table Bronze vers le snapshot après Batch 2...
Données à restaurer: 6,865 enregistrements

✅ Rollback Bronze terminé!


In [28]:
# Vérifier l'état après le rollback Bronze
print("=== ÉTAT APRÈS ROLLBACK BRONZE ===")

bronze_after_rollback = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"Bronze: {bronze_after_rollback:,} enregistrements")
print(f"(Avant rollback: {bronze_count:,})")
print(f"Différence: {bronze_count - bronze_after_rollback:,} enregistrements supprimés")

# Vérifier quels batchs sont présents
print("\nBatches présents après rollback:")
spark.sql("""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.bronze.sales
    GROUP BY batch_id
    ORDER BY batch_id
""").show()

=== ÉTAT APRÈS ROLLBACK BRONZE ===
Bronze: 6,865 enregistrements
(Avant rollback: 10,365)
Différence: 3,500 enregistrements supprimés

Batches présents après rollback:
+--------+-----+
|batch_id|count|
+--------+-----+
| batch_1| 3364|
| batch_2| 3501|
+--------+-----+



In [30]:
# ROLLBACK complet: Silver et Gold aussi
print("=== ROLLBACK des tables Silver et Gold ===")

# Rollback Silver avec le bon snapshot
silver_batch2 = spark.sql(f"""
    SELECT * FROM nessie.silver.sales VERSION AS OF '{batch2_snapshot_silver}'
""")

silver_batch2.writeTo("nessie.silver.sales").using("iceberg").createOrReplace()

print("✅ Rollback Silver terminé!")

# Rollback Gold avec le snapshot Gold de batch 2
gold_batch2 = spark.sql(f"""
    SELECT * FROM nessie.gold.sales_by_category_region VERSION AS OF '{batch2_snapshot_gold}'
""")

gold_batch2.writeTo("nessie.gold.sales_by_category_region").using("iceberg").createOrReplace()

print("✅ Rollback Gold terminé!")

=== ROLLBACK des tables Silver et Gold ===
✅ Rollback Silver terminé!
✅ Rollback Gold terminé!


In [35]:
# Vérifier l'état après le rollback Silver et Gold
print("=== ÉTAT APRÈS ROLLBACK SILVER ET GOLD ===\n")

silver_after_rollback = spark.sql("SELECT COUNT(*) FROM nessie.silver.sales").first()[0]
gold_after_rollback = spark.sql("SELECT COUNT(*) FROM nessie.gold.sales_by_category_region").first()[0]

print(f"Silver: {silver_after_rollback:,} enregistrements")
print(f"(Avant rollback: {silver_count:,})")
print(f"Différence: {silver_count - silver_after_rollback:,} enregistrements supprimés\n")

print(f"Gold: {gold_after_rollback:,} agrégats")
print(f"(Avant rollback: {gold_count:,})")

# Vérifier quels batchs sont présents dans Silver
print("\nBatches présents dans Silver après rollback:")
spark.sql("""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.silver.sales
    GROUP BY batch_id
    ORDER BY batch_id
""").show()

# Afficher un aperçu des données Gold pour vérifier
print("Aperçu des données Gold après rollback:")
spark.sql("""
    SELECT category, region, total_sales, order_count
    FROM nessie.gold.sales_by_category_region
    ORDER BY total_sales DESC
    LIMIT 5
""").show(truncate=False)

print("✅ Tous les rollbacks sont terminés et vérifiés!")
print("Les données sont revenue à l'état après Batch 2.")

=== ÉTAT APRÈS ROLLBACK SILVER ET GOLD ===

Silver: 6,587 enregistrements
(Avant rollback: 9,917)
Différence: 3,330 enregistrements supprimés

Gold: 12 agrégats
(Avant rollback: 12)

Batches présents dans Silver après rollback:
+--------+-----+
|batch_id|count|
+--------+-----+
| batch_1| 3260|
| batch_2| 3327|
+--------+-----+

Aperçu des données Gold après rollback:
+---------------+------+-----------+-----------+
|category       |region|total_sales|order_count|
+---------------+------+-----------+-----------+
|Furniture      |West  |180579.27  |475        |
|Technology     |East  |173185.88  |342        |
|Technology     |West  |154526.5   |382        |
|Furniture      |East  |145835.02  |397        |
|Office Supplies|East  |139495.68  |1157       |
+---------------+------+-----------+-----------+

✅ Tous les rollbacks sont terminés et vérifiés!
Les données sont revenue à l'état après Batch 2.


---
## BRANCHING NESSIE - VERSIONNING GIT-LIKE POUR LES DATA LAKES

### Concept

**Nessie** apporte le versionning de type Git pour les tables du data lake. Comme Git pour le code, Nessie permet :

* de créer des **branches** isolées pour expérimenter
* de tester des transformations de données sans impacter la production
* de **fusionner (merge)** les modifications validées dans la branche principale

### Cas d'usage

Un data engineer veut tester une nouvelle transformation ou ingérer un nouveau batch de données **sans risquer** de cycler la production. Il crée une branche de travail, fait ses tests, puis merge si tout est OK.

**Analogie Git** :
```
┌─────────┐         ┌─────────┐
│  main   │ ←------ │   dev   │
│ (prod)  │  merge  │ (test)  │
└─────────┘         └─────────┘
```

#### 1. Lister les références (branches) existantes

**Commande**: `LIST REFERENCES IN nessie`

Cette commande affiche toutes les branches présentes dans le catalogue Nessie. Par défaut, seule la branche `main` existe.

In [62]:
# Nettoyer la branche dev si elle existe pour une démo propre
spark.sql("DROP BRANCH IF EXISTS dev IN nessie").show()

# Afficher les branches Nessie
print("=== 1. Lister les références (branches) Nessie ===\n")
spark.sql("LIST REFERENCES IN nessie").show(truncate=False)

print("Explication:")
print("- Affiche toutes les branches du catalogue Nessie")
print("- Par défaut, seule 'main' existe (branche de production)")

+------+
|status|
+------+
|    OK|
+------+

=== 1. Lister les références (branches) Nessie ===

+-------+----+----------------------------------------------------------------+
|refType|name|hash                                                            |
+-------+----+----------------------------------------------------------------+
|Branch |main|de5b07ace1d6ab22451af012df0a5803c615147b0ba4dae2176a1d6f701e4cf0|
+-------+----+----------------------------------------------------------------+

Explication:
- Affiche toutes les branches du catalogue Nessie
- Par défaut, seule 'main' existe (branche de production)


#### 2. Voir la branche active

**Commande**: `SHOW REFERENCE IN nessie`

Cette commande affiche la branche sur laquelle nous sommes actuellement positionnés. Toutes les opérations SQL seront effectuées sur cette branche.

In [68]:
# Basculer sur la branche main
spark.sql("USE REFERENCE main IN nessie")

print("=== 2. Voir la branche active ===")
spark.sql("SHOW REFERENCE IN nessie").show(truncate=False)

current_ref = spark.sql("SHOW REFERENCE IN nessie").first()[0]
print(f"Nous sommes sur la branche: {current_ref}")
print("Toutes les opérations SQL affecteront cette branche.")

=== 2. Voir la branche active ===
+-------+----+----------------------------------------------------------------+
|refType|name|hash                                                            |
+-------+----+----------------------------------------------------------------+
|Branch |main|de5b07ace1d6ab22451af012df0a5803c615147b0ba4dae2176a1d6f701e4cf0|
+-------+----+----------------------------------------------------------------+

Nous sommes sur la branche: Branch
Toutes les opérations SQL affecteront cette branche.


#### 3. Créer une branche de travail

**Commande**: `CREATE BRANCH dev IN nessie FROM main`

Crée une nouvelle branche nommée `dev` à partir de `main`. Cette branche est initialement identique à `main` (copie instantanée, pas de copie de données).

In [70]:
print("=== 3. Créer une branche de travail 'dev' ===")

try:
    spark.sql("CREATE BRANCH dev IN nessie FROM main").show(truncate=False)
    print("✅ Branche 'dev' créée avec succès!")
except Exception as e:
    if "already exists" in str(e).lower():
        print("ℹ️  La branche 'dev' existe déjà (créée précédemment)")
    else:
        print(f"Info: {e}")

print("💡 Explication:")
print("- La branche 'dev' est une copie logique de 'main'")
print("- Pas de copie physique des données (instantané)")
print("- Les modifications sur 'dev' n'impacteront pas 'main'")

=== 3. Créer une branche de travail 'dev' ===
ℹ️  La branche 'dev' existe déjà (créée précédemment)
💡 Explication:
- La branche 'dev' est une copie logique de 'main'
- Pas de copie physique des données (instantané)
- Les modifications sur 'dev' n'impacteront pas 'main'


#### 4. Basculer sur la branche dev

**Commande**: `USE REFERENCE dev IN nessie`

Change la branche active pour `dev`. Maintenant, toutes les opérations (INSERT, UPDATE, CREATE TABLE) seront effectuées sur cette branche, pas sur `main`.

In [72]:
print("=== 4. Basculer sur la branche 'dev' ===")

spark.sql("USE REFERENCE dev IN nessie").show(truncate=False)

# Vérifier que nous sommes bien sur dev
spark.sql("SHOW REFERENCE IN nessie").show(truncate=False)

print("💡 Explication:")
print("- Nous sommes maintenant sur la branche 'dev'")
print("- Toutes les prochaines opérations affecteront 'dev'")
print("- La branche 'main' reste intacte (production safe)")

=== 4. Basculer sur la branche 'dev' ===
+-------+----+----------------------------------------------------------------+
|refType|name|hash                                                            |
+-------+----+----------------------------------------------------------------+
|Branch |dev |de5b07ace1d6ab22451af012df0a5803c615147b0ba4dae2176a1d6f701e4cf0|
+-------+----+----------------------------------------------------------------+

+-------+----+----------------------------------------------------------------+
|refType|name|hash                                                            |
+-------+----+----------------------------------------------------------------+
|Branch |dev |de5b07ace1d6ab22451af012df0a5803c615147b0ba4dae2176a1d6f701e4cf0|
+-------+----+----------------------------------------------------------------+

💡 Explication:
- Nous sommes maintenant sur la branche 'dev'
- Toutes les prochaines opérations affecteront 'dev'
- La branche 'main' reste intacte (producti

#### 5. Faire une modification sur la branche dev

Maintenant que nous sommes sur `dev`, ajoutons le Batch 3 pour tester notre pipeline. Cette modification n'impactera pas `main`.

In [73]:
print("=== 5. Modification sur la branche 'dev' (Ajout du Batch 3) ===\n")

# Vérifier l'état actuel sur dev
count_dev_before = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"Données Bronze sur 'dev' avant: {count_dev_before:,} enregistrements")

# Lire et ajouter le Batch 3 sur la branche dev
batch3_path = f"s3a://{AWS_BUCKET}/raw/batches/sales_batch_003.csv"

batch3_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(batch3_path)
)

batch3_bronze = (
    batch3_raw
    .withColumn("ingestion_date", current_date())
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", lit("sales_batch_3.csv"))
    .withColumn("source_system", lit("sales_system"))
    .withColumn("batch_id", lit("batch_3"))
)

batch3_bronze.writeTo("nessie.bronze.sales").append()

count_dev_after = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"Données Bronze sur 'dev' après:  {count_dev_after:,} enregistrements")
print(f"Différence: +{count_dev_after - count_dev_before:,} enregistrements")

print("\n💡 Cette modification est uniquement sur 'dev' - 'main' n'est pas impacté!")

=== 5. Modification sur la branche 'dev' (Ajout du Batch 3) ===

Données Bronze sur 'dev' avant: 6,865 enregistrements
Données Bronze sur 'dev' après:  10,365 enregistrements
Différence: +3,500 enregistrements

💡 Cette modification est uniquement sur 'dev' - 'main' n'est pas impacté!


#### 6. Comparer main vs dev

Vérifions que `main` n'a pas été modifié en comparant les données des deux branches.

In [74]:
print("=== 6. Comparaison: main vs dev ===\n")

# Données sur dev (où nous sommes)
dev_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]

# Basculer vers main pour comparer
spark.sql("USE REFERENCE main IN nessie")
main_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]

print(f"📊 BRANCHE 'dev' (avec Batch 3):  {dev_count:,} enregistrements")
print(f"📊 BRANCHE 'main' (production):     {main_count:,} enregistrements")
print(f"\n🔍 Différence: {dev_count - main_count:,} enregistrements de plus sur 'dev'")

# Afficher les batches présents sur main
print("\nBatches présents sur 'main':")
spark.sql("""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.bronze.sales
    GROUP BY batch_id
    ORDER BY batch_id
""").show()

print("\n💡 La branche 'main' est intacte! Les modifications sur 'dev'")
print("   n'ont pas impacté la production.")

=== 6. Comparaison: main vs dev ===

📊 BRANCHE 'dev' (avec Batch 3):  10,365 enregistrements
📊 BRANCHE 'main' (production):     6,865 enregistrements

🔍 Différence: 3,500 enregistrements de plus sur 'dev'

Batches présents sur 'main':
+--------+-----+
|batch_id|count|
+--------+-----+
| batch_1| 3364|
| batch_2| 3501|
+--------+-----+


💡 La branche 'main' est intacte! Les modifications sur 'dev'
   n'ont pas impacté la production.


#### 7. Afficher l'historique des commits

**Commande**: `SHOW LOG IN nessie`

Affiche l'historique des commits sur la branche active, similaire à `git log`. On voit toutes les opérations effectuées avec leurs métadonnées.

In [75]:
print("=== 7. Historique des commits sur 'main' ===\n")

spark.sql("SHOW LOG IN nessie").show(truncate=False)

print("\n💡 Explication:")
print("   - Chaque opération (CREATE, INSERT, MERGE) crée un commit")
print("   - Les commits ont un ID, un auteur, un timestamp et un message")
print("   - Comme Git, on peut naviguer dans l'historique!")

=== 7. Historique des commits sur 'main' ===

+------+---------+----------------------------------------------------------------+----------------------------------------------------+-----------+--------------------------+--------------------------+-----------------------------------------------------------------------------------------+
|author|committer|hash                                                            |message                                             |signedOffBy|authorTime                |committerTime             |properties                                                                               |
+------+---------+----------------------------------------------------------------+----------------------------------------------------+-----------+--------------------------+--------------------------+-----------------------------------------------------------------------------------------+
|enese |         |de5b07ace1d6ab22451af012df0a5803c615147b0ba4dae2176a1d6f7

#### 8. Fusionner la branche dev dans main

**Commande**: `MERGE BRANCH dev INTO main IN nessie`

Une fois les tests validés sur `dev`, on fusionne la branche dans `main`. Cela applique toutes les modifications de `dev` à la production.

In [76]:
print("=== 8. Fusionner 'dev' dans 'main' ===\n")

print("L'expérimentation sur 'dev' est réussie!")
print("On fusionne la branche dans 'main' pour déployer en production.\n")

spark.sql("MERGE BRANCH dev INTO main IN nessie").show(truncate=False)

print("\n✅ Fusion réussie! Les modifications de 'dev' sont maintenant dans 'main'.")

# Vérifier que main a maintenant les données
main_count_after = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"\n📊 'main' après fusion: {main_count_after:,} enregistrements")
print(f"   (avant fusion: {main_count:,}, +{main_count_after - main_count:,} nouveaux)")

# Afficher les batches présents sur main après fusion
print("\nBatches présents sur 'main' après fusion:")
spark.sql("""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.bronze.sales
    GROUP BY batch_id
    ORDER BY batch_id
""").show()

print("\n💡 Le Batch 3 est maintenant présent en production!")

=== 8. Fusionner 'dev' dans 'main' ===

L'expérimentation sur 'dev' est réussie!
On fusionne la branche dans 'main' pour déployer en production.

+----+----------------------------------------------------------------+
|name|hash                                                            |
+----+----------------------------------------------------------------+
|main|6180c28fc6aea5b5169ad772a66b0c9bb8a67b18b660e076770cf2c9083192ba|
+----+----------------------------------------------------------------+


✅ Fusion réussie! Les modifications de 'dev' sont maintenant dans 'main'.

📊 'main' après fusion: 10,365 enregistrements
   (avant fusion: 6,865, +3,500 nouveaux)

Batches présents sur 'main' après fusion:
+--------+-----+
|batch_id|count|
+--------+-----+
| batch_1| 3364|
| batch_2| 3501|
| batch_3| 3500|
+--------+-----+


💡 Le Batch 3 est maintenant présent en production!


### Résumé des commandes Nessie

| Commande | Description |
|----------|-------------|
| `LIST REFERENCES IN nessie` | Lister toutes les branches |
| `SHOW REFERENCE IN nessie` | Voir la branche active |
| `CREATE BRANCH nom IN nessie FROM main` | Créer une nouvelle branche |
| `USE REFERENCE nom IN nessie` | Changer de branche active |
| `SHOW LOG IN nessie` | Voir l'historique des commits |
| `MERGE BRANCH source INTO target IN nessie` | Fusionner une branche |

### Avantages du branching Nessie

* **Isolation** : Tester des transformations sans risque pour la production
* **Collaboration** : Plusieurs équipes peuvent travailler sur des branches parallèles
* **Audit** : Historique complet de toutes les modifications
* **Rollback** : Revenir à un état précédent si nécessaire

**Fin de la démo Branching!** 🎉